# Reading Stanford Digital Repository cloud formats with Python

This notebook uses each fixture PURL to discover the current file name in the Stanford Digital Repository (SDR). It then reads or displays the data directly from a Stacks URL.

Some fixtures are restricted. Those cells are included for completeness, but they require an authorized SDR session.

## Image - single image
PURL: https://purl.stanford.edu/bc151bq1744

In [ ]:
# Install the libraries used to request and display the IIIF image.
%pip -q install requests pillow

# Import the library used to request the image.
import requests

# Import the notebook display helper.
from IPython.display import display

# Import Pillow's image reader.
from PIL import Image

# Import an in-memory byte stream helper.
from io import BytesIO

# Set the 1-based image number to retrieve.
image_index = 48

# Define the SDR IIIF URL with a maximum width of 1200 pixels.
image_url = (
    f"https://stacks.stanford.edu/image/iiif/"
    f"bc151bq1744%2Fbc151bq1744_00_{image_index:04d}/full/800,/0/default.jpg"
)

# Request the resized image from SDR.
response = requests.get(image_url)

# Stop with a useful error if SDR does not return the image.
response.raise_for_status()

# Open the image from the downloaded bytes.
image = Image.open(BytesIO(response.content))

# Display the IIIF image at its returned width, no larger than 800 pixels.
display(image)

## Image - multi-image
PURL: https://purl.stanford.edu/bc151bq1744

The fixture uses the same PURL as the single-image example. The IIIF manifest may contain multiple canvases; this minimal example previews the first one.

In [ ]:
# Install the libraries used to request and display IIIF images.
%pip -q install requests pillow

# Import the library used to request the manifest and images.
import requests

# Import helpers for downloading thumbnails in parallel.
from concurrent.futures import ThreadPoolExecutor, as_completed

# Import the notebook display helper.
from IPython.display import display

# Import Pillow's image reader and thumbnail helpers.
from PIL import Image, ImageDraw, ImageFont, ImageOps

# Import an in-memory byte stream helper.
from io import BytesIO

# Import the ceiling and square root helpers used to size the thumbnail grid.
from math import ceil, sqrt

# Define the SDR IIIF Presentation manifest URL.
manifest_url = "https://purl.stanford.edu/bc151bq1744/iiif/manifest"

# Reuse one connection pool for every request in this cell.
session = requests.Session()

# Cap how long any single request may hang before failing, in seconds.
request_timeout = 10

# Request the multi-image manifest from SDR.
manifest_response = session.get(manifest_url, timeout=request_timeout)
manifest_response.raise_for_status()
manifest = manifest_response.json()
canvases = manifest["sequences"][0]["canvases"]

print(f"This manifest contains {len(canvases)} images.")

# Choose which 0-based image indexes to preview, e.g. range(0, 11) or [0, 45, 90].
preview_indices = range(0, 11)

# Keep only valid, unique indexes while preserving ascending order.
selected_indices = sorted({index for index in preview_indices if 0 <= index < len(canvases)})
print(f"Previewing {len(selected_indices)} of {len(canvases)} images: {selected_indices}")

# Reserve a fixed tile for each thumbnail and its 1-based image index.
tile_width = 220
tile_height = 250
image_area_height = 215
label_area_height = tile_height - image_area_height

# Lay the thumbnails out in a roughly square grid sized to the selected images.
grid_columns = ceil(sqrt(len(selected_indices)))
grid_rows = ceil(len(selected_indices) / grid_columns)
grid_image = Image.new("RGB", (grid_columns * tile_width, grid_rows * tile_height), "white")
grid_display = display(grid_image, display_id=True)

# Use Pillow's built-in font so the cell does not need another dependency.
label_font = ImageFont.load_default()

def draw_tile(index, thumbnail):
    tile = Image.new("RGB", (tile_width, tile_height), "white")
    tile_draw = ImageDraw.Draw(tile)

    if thumbnail is not None:
        thumbnail = thumbnail.convert("RGB")
        fitted_thumbnail = ImageOps.contain(thumbnail, (tile_width, image_area_height))
        image_x = (tile_width - fitted_thumbnail.width) // 2
        image_y = (image_area_height - fitted_thumbnail.height) // 2
        tile.paste(fitted_thumbnail, (image_x, image_y))
    else:
        tile_draw.text((10, image_area_height // 2), "Unavailable", fill="gray")

    label = f"Image {index + 1}"
    label_box = tile_draw.textbbox((0, 0), label, font=label_font)
    label_x = (tile_width - (label_box[2] - label_box[0])) // 2
    label_y = image_area_height + (label_area_height - (label_box[3] - label_box[1])) // 2
    tile_draw.text((label_x, label_y), label, fill="black", font=label_font)
    return tile


def fetch_thumbnail(index_and_canvas):
    index, canvas = index_and_canvas
    service_url = canvas["images"][0]["resource"]["service"]["@id"]
    thumbnail_url = f"{service_url}/full/{tile_width},/0/default.jpg"
    try:
        response = session.get(thumbnail_url, timeout=request_timeout)
        response.raise_for_status()
    except requests.RequestException as error:
        print(f"Thumbnail {index + 1} failed: {error}")
        return index, None
    return index, Image.open(BytesIO(response.content))

max_workers = 8
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [
        executor.submit(fetch_thumbnail, (index, canvases[index]))
        for index in selected_indices
    ]
    for future in as_completed(futures):
        index, thumbnail = future.result()
        position = selected_indices.index(index)
        column, row = position % grid_columns, position // grid_columns
        grid_image.paste(draw_tile(index, thumbnail), (column * tile_width, row * tile_height))
        grid_display.update(grid_image)

# Preview the last selected image at full size.
page_index = selected_indices[-1]
selected_service_url = canvases[page_index]["images"][0]["resource"]["service"]["@id"]
selected_image_url = f"{selected_service_url}/full/1200,/0/default.jpg"
selected_response = session.get(selected_image_url, timeout=request_timeout)
selected_response.raise_for_status()
selected_image = Image.open(BytesIO(selected_response.content))
display(selected_image)

## Index map
PURL: https://purl.stanford.edu/kh860qj3406
Format: Shapefile

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR FlatGeobuf URL.
data_url = "https://stacks.stanford.edu/file/kh860qj3406/index_map.fgb"

# Create a world map.
map_view = Map(center=(0, 20), zoom=2)

# Add the cloud-optimized vector layer to the map.
map_view.add_flatgeobuf(data_url, name="SDR index map")

# Display the interactive map.
map_view

## Shapefile polygons
PURL: https://purl.stanford.edu/cr288qn9438

In [1]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR FlatGeobuf URL.
data_url = "https://stacks.stanford.edu/file/cr288qn9438/alameda2014.fgb"

# Create a map centered near California.
map_view = Map(center=(-121.8, 37.6), zoom=8)

# Add the cloud-optimized vector layer to the map.
map_view.add_flatgeobuf(data_url, name="SDR polygon data")

# Display the interactive map.
map_view

Note: you may need to restart the kernel to use updated packages.


## Shapefile polygons (restricted)
PURL: https://purl.stanford.edu/bb099zb1450
This item will require SDR authorization.

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR FlatGeobuf URL.
data_url = "https://stacks.stanford.edu/file/bb099zb1450/D31_dep.fgb"

# Create a map centered near France.
map_view = Map(center=(1.2, 43.4), zoom=8)

# Add the restricted cloud-optimized vector layer when authorized.
map_view.add_flatgeobuf(data_url, name="SDR restricted polygon data")

# Display the interactive map.
map_view

## Shapefile polygons (crosses 180 longitude)
PURL: https://purl.stanford.edu/gk831nh0251

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR FlatGeobuf URL.
data_url = "https://stacks.stanford.edu/file/gk831nh0251/KIR_adm0.fgb"

# Create a world map because the geometry crosses the 180 degree meridian.
map_view = Map(center=(180, 0), zoom=2)

# Add the cloud-optimized vector layer to the map.
map_view.add_flatgeobuf(data_url, name="SDR antimeridian polygon data")

# Display the interactive map.
map_view

## Shapefile points
PURL: https://purl.stanford.edu/bb434sw7474

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR FlatGeobuf URL.
data_url = "https://stacks.stanford.edu/file/bb434sw7474/gar_exp_SAU.fgb"

# Create a map centered near Saudi Arabia.
map_view = Map(center=(45, 24), zoom=5)

# Add the cloud-optimized point layer to the map.
map_view.add_flatgeobuf(data_url, name="SDR point data")

# Display the interactive map.
map_view

## Shapefile points (restricted)
PURL: https://purl.stanford.edu/mx498nz6026
This item may require SDR authorization.

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR FlatGeobuf URL.
data_url = "https://stacks.stanford.edu/file/mx498nz6026/ASSAM_POINT.fgb"

# Create a map centered near Assam, India.
map_view = Map(center=(93, 26), zoom=7)

# Add the restricted cloud-optimized point layer when authorized.
map_view.add_flatgeobuf(data_url, name="SDR restricted point data")

# Display the interactive map.
map_view

## Shapefile lines
PURL: https://purl.stanford.edu/zc952cn1722

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR FlatGeobuf URL.
data_url = "https://stacks.stanford.edu/file/zc952cn1722/Faults_OffshoreSantaBarbara.fgb"

# Create a map centered near Santa Barbara.
map_view = Map(center=(-119.7, 34.4), zoom=10)

# Add the cloud-optimized line layer to the map.
map_view.add_flatgeobuf(data_url, name="SDR line data")

# Display the interactive map.
map_view

## Shapefile lines (restricted)
PURL: https://purl.stanford.edu/wn662ny6315
This item may require SDR authorization.

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR FlatGeobuf URL.
data_url = "https://stacks.stanford.edu/file/wn662ny6315/DRAINAGE.fgb"

# Create a map centered near Rajasthan, India.
map_view = Map(center=(73, 26.7), zoom=7)

# Add the restricted cloud-optimized line layer when authorized.
map_view.add_flatgeobuf(data_url, name="SDR restricted drainage data")

# Display the interactive map.
map_view

## PMTiles many layers
PURL: https://purl.stanford.edu/hf224mw4004
PMTiles is a cloud-optimized tile archive. Leafmap requests only the tiles needed for the visible map.

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR PMTiles URL.
data_url = "https://stacks.stanford.edu/file/hf224mw4004/20231116.pmtiles"

# Create a world map.
map_view = Map(center=(-100, 40), zoom=2)

# Add the cloud-optimized tile archive to the map.
map_view.add_pmtiles(data_url, name="SDR PMTiles many layers")

# Display the interactive map.
map_view

## GeoJSON points
PURL: https://purl.stanford.edu/nz577hv6134

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR FlatGeobuf URL derived from the GeoJSON fixture.
data_url = "https://stacks.stanford.edu/file/nz577hv6134/samtrans_stops_20150626_20260403.fgb"

# Create a map centered near San Mateo County.
map_view = Map(center=(-122.3, 37.5), zoom=10)

# Add the cloud-optimized point layer to the map.
map_view.add_flatgeobuf(data_url, name="SDR GeoJSON points as FlatGeobuf")

# Display the interactive map.
map_view

## GeoJSON lines
PURL: https://purl.stanford.edu/yt111kw1413

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR FlatGeobuf URL derived from the GeoJSON fixture.
data_url = "https://stacks.stanford.edu/file/yt111kw1413/samTrans_bus_routes_20151021_shapes_20260406.fgb"

# Create a map centered near San Mateo County.
map_view = Map(center=(-122.3, 37.5), zoom=10)

# Add the cloud-optimized line layer to the map.
map_view.add_flatgeobuf(data_url, name="SDR GeoJSON lines as FlatGeobuf")

# Display the interactive map.
map_view

## GeoJSON polygons
PURL: https://purl.stanford.edu/ch787ty4618

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR FlatGeobuf URL derived from the GeoJSON fixture.
data_url = "https://stacks.stanford.edu/file/ch787ty4618/County_Boundary.fgb"

# Create a map centered near San Mateo County.
map_view = Map(center=(-122.3, 37.5), zoom=10)

# Add the cloud-optimized polygon layer to the map.
map_view.add_flatgeobuf(data_url, name="SDR GeoJSON polygons as FlatGeobuf")

# Display the interactive map.
map_view

## GeoJSON points (slow to load)
PURL: https://purl.stanford.edu/qp917dm2243

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR FlatGeobuf URL derived from the large GeoJSON fixture.
data_url = "https://stacks.stanford.edu/file/qp917dm2243/Stanford_Temperature_Model_0km.fgb"

# Create a map centered on the conterminous United States.
map_view = Map(center=(-96, 38), zoom=4)

# Add the cloud-optimized point layer to the map.
map_view.add_flatgeobuf(data_url, name="SDR large GeoJSON as FlatGeobuf")

# Display the interactive map.
map_view

## GeoTIFF (EPSG::32735)
PURL: https://purl.stanford.edu/bb223nv3920
The helper selects the SDR Cloud Optimized GeoTIFF derivative and reads it by HTTP range requests.

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR Cloud Optimized GeoTIFF URL.
data_url = "https://stacks.stanford.edu/file/bb223nv3920/Pretoria_landcover_t1_cog.tif"

# Create a map centered near Pretoria, South Africa.
map_view = Map(center=(28, -25.5), zoom=8)

# Add the COG with a color palette for raster values.
map_view.add_cog(data_url, name="SDR COG EPSG:32735", colormap="viridis")

# Display the interactive map.
map_view

## GeoTIFF (EPSG::4326, restricted)
PURL: https://purl.stanford.edu/bb021mm7809
This item may require SDR authorization.

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct restricted SDR Cloud Optimized GeoTIFF URL.
data_url = "https://stacks.stanford.edu/file/bb021mm7809/MCE_FI2G_2014_cog.tif"

# Create a world map because the dataset covers Finland.
map_view = Map(center=(25, 65), zoom=5)

# Add the restricted COG when the user is authorized.
map_view.add_cog(data_url, name="SDR restricted COG EPSG:4326", colormap="viridis")

# Display the interactive map.
map_view

## GeoTIFF (EPSG::4326, nodata configured)
PURL: https://purl.stanford.edu/kq996gp6880

In [ ]:
# Install GeoLibre for an interactive notebook map.
%pip install geolibre

# Import the GeoLibre map widget.
from geolibre import Map

# Define the direct SDR Cloud Optimized GeoTIFF URL.
data_url = "https://stacks.stanford.edu/file/kq996gp6880/SF1938_cog.tif"

# Create a map centered on San Francisco.
map_view = Map(center=(-122.44, 37.76), zoom=11)

# Add the COG with a color palette for raster values.
map_view.add_cog(data_url, name="SDR nodata COG EPSG:4326", colormap="viridis")

# Display the interactive map.
map_view

## IIIF map with georeference annotation
PURL: https://purl.stanford.edu/bb013fz9675

This example uses Allmaps' hosted annotations API (`https://annotations.allmaps.org/?url={manifest_url}`) to look up the georeference annotation for the IIIF manifest. Allmaps also serves XYZ tiles for any annotation it hosts, which this cell renders on a GeoLibre map. An SDR-hosted annotation and XYZ tile service would be the preferred long-term Earthworks infrastructure, removing the dependency on Allmaps' hosted API.


In [6]:
# Install GeoLibre for an interactive notebook map.
%pip -q install geolibre requests

# Import the library used to request JSON from public web services.
import requests

# Import the notebook display helper for a clickable Allmaps Editor link.
from IPython.display import display, Markdown

# Import the GeoLibre map widget.
from geolibre import Map

# Define the SDR IIIF manifest URL for this item.
manifest_url = "https://purl.stanford.edu/bb013fz9675/iiif/manifest"

# Ask Allmaps' hosted API for any georeference annotation it has for this manifest.
annotation_api_url = f"https://annotations.allmaps.org/?url={manifest_url}"

# Request the georeference annotation page from Allmaps.
annotation_response = requests.get(annotation_api_url)

# Stop with a useful error if Allmaps itself is unreachable.
annotation_response.raise_for_status()

# Read any georeference annotations Allmaps already has for this manifest.
annotations = annotation_response.json().get("items", [])

if not annotations:
    # Build a clickable Allmaps Editor link so a new georeferencing session can be started.
    editor_url = f"https://editor.allmaps.org/images?url={manifest_url}"
    display(Markdown(
        f"No Allmaps georeference annotation was found for this manifest.\n\n"
        f"Start one in the [Allmaps Editor]({editor_url})."
    ))
else:
    # Read the first annotation's id, which embeds the Allmaps map ID.
    annotation_id = annotations[0]["id"]
    map_id = annotation_id.rsplit("/", 1)[-1]

    # Build the Allmaps XYZ tile URL template for the georeferenced map.
    tile_url = f"https://allmaps.xyz/maps/{map_id}/{{z}}/{{x}}/{{y}}.png"

    # Request the map's georeferenced footprint from Allmaps to zoom the view to it.
    map_info_response = requests.get(f"https://api.allmaps.org/maps/{map_id}")
    map_info_response.raise_for_status()
    # Prefer the map metadata's geoMask; if Allmaps has changed the response schema,
    # fall back to the annotation payload geometry.
    map_info = map_info_response.json()
    geo_mask = None

    if isinstance(map_info, dict):
        geo_mask = (
            map_info.get("geoMask")
            or map_info.get("geo_mask")
            or map_info.get("geometry")
        )
        if geo_mask is None and isinstance(map_info.get("features"), list):
            for feature in map_info["features"]:
                if isinstance(feature, dict):
                    geo_mask = feature.get("geometry")
                    if geo_mask is not None:
                        break

    if geo_mask is None:
        for payload in (annotation, annotation_page):
            if not isinstance(payload, dict):
                continue
            geo_mask = payload.get("geoMask") or payload.get("geometry")
            if geo_mask is not None:
                break
            items = payload.get("items") or []
            for item in items:
                if not isinstance(item, dict):
                    continue
                geo_mask = item.get("geoMask") or item.get("geometry")
                if geo_mask is None and isinstance(item.get("body"), dict):
                    geo_mask = item["body"].get("geoMask") or item["body"].get("geometry")
                if geo_mask is not None:
                    break
            if geo_mask is not None:
                break

    if geo_mask is None:
        raise KeyError("geoMask not found in the Allmaps response or annotation payload.")

    # Wrap the footprint geometry as a GeoJSON feature.
    geo_mask_feature = {"type": "Feature", "properties": {}, "geometry": geo_mask}

    # Compute the footprint's bounding box center so the map can zoom to the image.
    lons = [point[0] for point in geo_mask["coordinates"][0]]
    lats = [point[1] for point in geo_mask["coordinates"][0]]
    center = ((min(lons) + max(lons)) / 2, (min(lats) + max(lats)) / 2)

    # Create a map centered on the georeferenced image footprint.
    map_view = Map(center=center, zoom=14)

    # Add the Allmaps XYZ tile layer to the map.
    map_view.add_tile_layer(tile_url, name="Allmaps georeferenced map")

    # Add the annotation's footprint outline so the extent is visible alongside the tiles.
    map_view.add_geojson(geo_mask_feature, name="Allmaps footprint")

    # Display the interactive map.
    map_view


Note: you may need to restart the kernel to use updated packages.


KeyError: 'geoMask not found in the Allmaps response or annotation payload.'